<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/Notes/06-dictionaries_pandas/06_intro_to_dictionaries_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 6: Dictionaries and pandas

**DSC 210 Foundations of Data Science**

References:
- [Hands-on Introduction to Data Science with Python](https://florian-huber.github.io/data_science_course/) (CC BY-NC-SA 4.0)
- [pandas documentation](https://pandas.pydata.org/docs/)

```
ASK  ->  [ GET ]  ->  EXPLORE  ->  MODEL  ->  COMMUNICATE
```

*Last major revision: 2026-08-08*

So far we have used Python (Module 3), NumPy (Module 4), and seaborn (Module 5).

NumPy gave us fast arrays of numbers, but real datasets are **tables**: many rows, several columns, and a mix of text and numbers in each row.

This module gives us two tools that carry us through the **GET** stage and into **EXPLORE**: the Python *dictionary* (a labeled lookup) and the pandas *DataFrame* (a spreadsheet in code).

## Key Concepts

- Store labeled data in a **dictionary**, and add, modify, delete, and iterate over key-value pairs
- Recognize a *record* as a dictionary and a *table* as a collection of records
- Build a pandas **DataFrame** from dictionaries, and read one from a CSV file
- Inspect, select, filter, and sort a DataFrame, and create new columns
- Compute descriptive statistics, including one summary per group with `groupby`
- Recognize that a derived column can rank the same data two different ways, and say which ranking answers the question

**A note on the code cells.** Cells marked `# RUN-TOGETHER` are complete and run as written. Cells marked `# FILL-IN` contain blanks written as `____` and will raise a `SyntaxError` until every blank is replaced.

---
## 1. Dictionaries
---
### 1.1 The lookup problem

Suppose the campus radio station is tracking how many times some popular songs have been streamed (in billions). A first instinct, using what we know from Module 3, is two parallel lists:

`titles  = ['Blinding Lights', 'Shape of You', 'Riptide']`

`streams = [5.3, 4.8, 3.5]`

Simple question: *How many streams does 'Riptide' have?* With lists, we first have to find where the title lives, then read the matching position in the other list.

In [ ]:
# RUN-TOGETHER
titles  = ['Blinding Lights', 'Shape of You', 'Riptide']
streams = [5.3, 4.8, 3.5]

# To look up Riptide we must find its position first, then index the OTHER list.
idx = titles.index('Riptide')     # position of the title
print('position:', idx)
print('streams (billions):', streams[idx])

This works, but it is fragile:

- The two lists must stay in the **same order** forever. If we sort one and forget the other, every lookup silently returns the wrong song.
- Looking something up takes **two steps** (find the position, then index).

We want to ask for a value by its name, in one step. That is exactly what a **dictionary** does.

### 1.2 Dictionaries: lookup by key

**Definition.** A *dictionary* is an unordered collection of **key-value pairs**, written in curly braces `{ }`. Each **key** points to a **value**, and we retrieve a value by writing the key in square brackets: `d[key]`.

- Keys must be unique and immutable (strings, ints, floats, booleans). Text keys are the most common.
- Values can be anything: numbers, strings, lists, or even other dictionaries.

Instead of two lists that must stay aligned, we bind each title to its streams figure directly.

In [ ]:
# RUN-TOGETHER
# Curly braces, and 'key : value' pairs separated by commas.
song_streams = {'Blinding Lights': 5.3,
                'Shape of You': 4.8,
                'Riptide': 3.5}

# One-step lookup by name: no positions, no second list.
print(song_streams['Riptide'])

print(song_streams)
print(type(song_streams))

#### **Activity 6.1 - How far ahead is the leader?**

**A.** Using two lookups and subtraction, print how many billion streams `'Blinding Lights'` is ahead of `'Riptide'`.

**B.** Predict, in a comment: what does Python do if you ask for a title that is *not* a key, such as `song_streams['Levitating']`? We fix this safely in the next section.

In [ ]:
# FILL-IN  (Activity 6.1)
# A. How far ahead is Blinding Lights over Riptide?
gap = song_streams[____] - song_streams[____]
print('Blinding Lights leads Riptide by', round(gap, 1), 'billion streams')

# B. PREDICT (comment only, do not run a missing-key lookup yet):
#    song_streams['Levitating'] would do what?
#    my guess: ____

### 1.3 Adding, modifying, and deleting pairs

A dictionary is *changeable*. We use the same square-bracket syntax to store a value:

- **Add** a new pair: `d[new_key] = value`
- **Modify** an existing value: `d[existing_key] = new_value` (same syntax, an existing key)
- **Check membership**: `key in d` returns `True`/`False`
- **Delete** a pair: `del d[key]`

Real data arrives incrementally and gets corrected. Adding, fixing, and removing entries is the day-to-day reality of maintaining a dataset.

In [ ]:
# RUN-TOGETHER
# ADD a song the station just started tracking.
song_streams['Die With a Smile'] = 3.5
print('after adding:', song_streams)

# MEMBERSHIP test: is a key present?
print("'Die With a Smile' in song_streams? ->", 'Die With a Smile' in song_streams)
print("'Levitating' in song_streams?       ->", 'Levitating' in song_streams)

# MODIFY: streams keep climbing.
song_streams['Die With a Smile'] = 3.6
print('after updating:', song_streams['Die With a Smile'])

# DELETE: the station stops tracking Shape of You.
del song_streams['Shape of You']
print('after deleting:', song_streams)

#### **Activity 6.2 - Corrections, not overwrites**

Real datasets get *corrected*, not just overwritten. Working from the current `song_streams`:

**A.** The station realizes `'Blinding Lights'` was over-counted by `0.4`. Lower it using its own current value: read the value, subtract, store it back. Do not type the new number.

**B.** Add `'Heat Waves'` at `3.7`.

**C.** Complete the `if`/`else` so it prints a song's streams *only if* the title is present, and otherwise prints "not tracked". Test it on `'Yellow'`, which is absent.

In [ ]:
# FILL-IN  (Activity 6.2)
# A. read-modify-write: lower Blinding Lights by 0.4 using its CURRENT value
song_streams['Blinding Lights'] = song_streams[____] ____ 0.4

# B. add Heat Waves at 3.7
song_streams[____] = ____

# C. safe lookup guarded by `in`
title = 'Yellow'
if title ____ song_streams:
    print(title, '->', song_streams[title])
else:
    print(title, 'is not tracked')

print(song_streams)

### 1.4 Iterating over a dictionary

Often we want to *walk through every pair* to print a report, add things up, or transform the data. Three views help:

- `d.keys()`   -> the keys
- `d.values()` -> the values
- `d.items()`  -> the (key, value) pairs

A `for` loop over `d.items()` hands us the key and value together on each pass.

In [ ]:
# RUN-TOGETHER
song_streams = {'Blinding Lights': 5.3,
                'Shape of You': 4.8,
                'Heat Waves': 3.7,
                'Riptide': 3.5,
                'Die With a Smile': 3.5}

for title, plays in song_streams.items():
    print(title, 'has about', plays, 'billion streams')

#### **Class Example 6.1 - Totaling and finding the maximum**

Because we can visit every value, we can also *summarize* the dictionary. Before running the code, let us trace the loop by hand.

**Step 0.** Initially `total = 0`, no `top_title`, and `top_plays = 0`.

**Step 1.** First pair: `title = 'Blinding Lights'`, `plays = 5.3`.
`total = 0 + 5.3 = 5.3`. Since `5.3 > 0`, set `top_plays = 5.3` and `top_title = 'Blinding Lights'`.

**Step 2.** Next pair: `title = 'Shape of You'`, `plays = 4.8`.
`total = 5.3 + 4.8 = 10.1`. Since `4.8` is not greater than `5.3`, the top is unchanged.

And so on: `total` keeps growing, and no later value exceeds 5.3.

In [ ]:
# RUN-TOGETHER
# initial state
total = 0
top_title = None
top_plays = 0

for title, plays in song_streams.items():
    total = total + plays               # running sum of plays
    if plays > top_plays:               # track the largest so far
        top_plays = plays
        top_title = title

print('total streams (billions):', round(total, 1))
print('most-streamed:', top_title, '(', top_plays, 'billion )')

### 1.5 A record is a dictionary

So far each value has been a single number. But a dictionary can describe one whole thing by using *different fields as keys*. This is called a **record**.

**Definition.** A *record* is a dictionary whose keys are **field names** (like `title`, `year`) and whose values are that one item's data. A record is the natural stand-in for one row of a table.

In [ ]:
# RUN-TOGETHER
# One song described as a record: several fields, mixed value types.
blinding_lights = {'title': 'Blinding Lights',
                   'artist': 'The Weeknd',
                   'year': 2019,
                   'genre': 'Synth-pop',
                   'streams_billions': 5.3}

print(blinding_lights['title'], 'was released in', blinding_lights['year'])
print('genre:', blinding_lights['genre'])

If one record is a row, then a **list of records is a table**. Here are three songs, each a record. Keep this object in mind: we convert it into a real table in Section 2.2.

In [ ]:
# RUN-TOGETHER
# A list of dictionaries: each dictionary is one row.
song_records = [
    {'title': 'Blinding Lights',  'artist': 'The Weeknd', 'year': 2019, 'streams_billions': 5.3},
    {'title': 'Riptide',          'artist': 'Vance Joy',  'year': 2013, 'streams_billions': 3.5},
    {'title': 'Bohemian Rhapsody','artist': 'Queen',      'year': 1975, 'streams_billions': 3.1},
]

# Reach into the second record (index 1), then the 'artist' field.
print(song_records[1]['title'], 'is by', song_records[1]['artist'])

#### **Activity 6.3 - Filtering a list of records**

Print the `title` of every song released in *2015 or later*.

In [ ]:
# FILL-IN  (Activity 6.3)
catalog = [
    {'title': 'Levitating', 'artist': 'Dua Lipa',    'year': 2020, 'streams_billions': 2.6},
    {'title': 'Yellow',     'artist': 'Coldplay',    'year': 2000, 'streams_billions': 3.6},
    {'title': 'Flowers',    'artist': 'Miley Cyrus', 'year': 2023, 'streams_billions': 2.8},
    {'title': 'Creep',      'artist': 'Radiohead',   'year': 1992, 'streams_billions': 2.7},
]

for song in catalog:
    if song[____] >= 2015:
        print(song[____])

### 1.6 Lists versus dictionaries: which one?

| | Indexed by | Order matters? | Good for |
| --- | --- | --- | --- |
| **List** | position (`0, 1, 2, ...`) | yes | a sequence of values (a column) |
| **Dictionary** | unique key | no | a fast lookup table; a record with named fields |

Rule of thumb: if you find yourself keeping two lists *in sync* so that position `i` in one matches position `i` in another, you probably want a dictionary, or, very soon, a DataFrame.

---
## 2. From Dictionaries to pandas DataFrames
---
### 2.1 Why pandas?

A list of records is a table *in spirit*, but doing table work with raw dictionaries (filtering rows, averaging a column) quickly gets clumsy. **pandas** is the standard Python library for tabular data.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/06-dictionaries_pandas/pandas.jpg?raw=true" width="400">

**Definition.** A *pandas DataFrame* is a two-dimensional table of data.
- Each **row** is one observation (here, one song).
- Each **column** is one feature (title, artist, year, ...), and has a label.
- It is built on top of NumPy, so column math is fast and element-wise, just like Module 4.

By convention we import it as `pd`.

**Note:** we have been using DataFrames since Module 2 without naming them. `sns.load_dataset('penguins')` returns a DataFrame.

In [ ]:
# RUN-TOGETHER
import pandas as pd

### 2.2 Building a DataFrame from a dictionary

There are two natural shapes, and pandas accepts both.

**Shape 1: a dictionary of columns.** Each **key** is a column name, and each **value** is a list holding that column's data, top to bottom.

In [ ]:
# RUN-TOGETHER
# A dictionary of COLUMNS: key = column name, value = list of column data.
data = {
    'title':  ['Blinding Lights', 'Riptide', 'Bohemian Rhapsody'],
    'artist': ['The Weeknd', 'Vance Joy', 'Queen'],
    'year':   [2019, 2013, 1975],
    'streams_billions': [5.3, 3.5, 3.1],
}

from_columns = pd.DataFrame(data)
print(from_columns)

#### **Class Example 6.2 - The other shape: a list of records**

Remember `song_records` from Section 1.5, the list of dictionaries where each dictionary was one row? Hand that straight to `pd.DataFrame` and pandas reads the keys as column names.

This is the bridge the whole first half of the module was building toward: **a list of records is a table**, and now it literally is one.

In [ ]:
# RUN-TOGETHER
from_records = pd.DataFrame(song_records)   # a LIST OF ROWS
print(from_records)

print()
print('same table both ways?', from_columns.equals(from_records))

> **Discuss.** Both shapes produced the identical table. When data arrives one observation at a time (a survey response, a sensor reading, an API result), which shape does it naturally arrive in? Which shape would a spreadsheet export give you?

### 2.3 Inspecting a dataset

Let's load the station's full list of most-streamed songs.

In [ ]:
# RUN-TOGETHER
songs_data = {
    'title': ['Blinding Lights', 'Shape of You', 'Perfect', 'Believer', 'Heat Waves',
              'Yellow', 'Riptide', 'Die With a Smile', 'Take Me to Church', 'Die For You',
              'Bohemian Rhapsody', 'Mr. Brightside'],
    'artist': ['The Weeknd', 'Ed Sheeran', 'Ed Sheeran', 'Imagine Dragons', 'Glass Animals',
               'Coldplay', 'Vance Joy', 'Lady Gaga & Bruno Mars', 'Hozier', 'The Weeknd',
               'Queen', 'The Killers'],
    'year': [2019, 2017, 2017, 2017, 2020, 2000, 2013, 2024, 2013, 2016, 1975, 2004],
    'genre': ['Synth-pop', 'Pop', 'Pop', 'Pop rock', 'Indie pop', 'Alternative rock',
              'Indie folk', 'Pop', 'Indie rock', 'R&B', 'Rock', 'Rock'],
    'streams_billions': [5.3, 4.8, 3.9, 3.8, 3.7, 3.6, 3.5, 3.5, 3.4, 3.2, 3.1, 3.0],
}

songs = pd.DataFrame(songs_data)
songs

When you meet a table, look before you leap. These are the first commands to run on any new DataFrame:

- `df.head(n)`  -> the first `n` rows (default 5)
- `df.shape`    -> `(rows, columns)`; an **attribute**, so no parentheses (recall Module 4)
- `df.columns`  -> the column labels
- `df.dtypes`   -> the type stored in each column
- `df.info()`   -> a compact summary: columns, non-null counts, dtypes
- `df.describe()` -> descriptive statistics for the numeric columns

In [ ]:
# RUN-TOGETHER
print('shape (rows, cols):', songs.shape)     # attribute: no ()
print()
print('column labels:', list(songs.columns))
print()
print('dtypes:')
print(songs.dtypes)

In [ ]:
# RUN-TOGETHER
songs.info()    # structure + non-null counts (great for spotting missing data)

In [ ]:
# RUN-TOGETHER
songs.describe()

### 2.4 Reading data from a CSV file

Typing a dataset by hand is fine for a class demo, but in practice data lives in files. The most common is a **CSV** (comma-separated values): a plain-text table where columns are separated by commas and each line is a row.

Doing the round trip once shows exactly what `read_csv` reads back. We save `songs` to a file, then read it into a fresh DataFrame.

In [ ]:
# RUN-TOGETHER
# index=False keeps pandas from writing the row numbers as a column.
songs.to_csv('songs_export.csv', index=False)

# Read it back. In real projects you START here, with a file someone handed you.
songs = pd.read_csv('songs_export.csv')

# ALWAYS check that it loaded the way you expected.
songs.head()

Files can also be read straight from the web, which is how the course datasets are stored.

In [ ]:
# RUN-TOGETHER
url = 'https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/csv_data/songs.csv'
songs_from_web = pd.read_csv(url)
print(songs_from_web.shape)
songs_from_web.head(3)

---
## 3. Exploring a DataFrame
---
With one clean table we can select, filter, sort, derive, and summarize. These are core moves of the EXPLORE stage.

### 3.1 Selecting columns

- **Single brackets** with one column name -> a Series (a labeled 1-D column).
- **Double brackets** with a list of names -> a DataFrame (a table, even for one column).

Think of it as: *one thing in the brackets, get a column; a list in the brackets, get a table.*

In [ ]:
# RUN-TOGETHER
# Single brackets -> Series
titles = songs['title']
print(type(titles))     # pandas Series (a labeled 1-D array)
print(titles.head(3))

print()
# Double brackets -> DataFrame (note the list inside)
subset = songs[['title', 'streams_billions']]
print(type(subset))
subset.head(3)

### 3.2 Selecting rows: `.iloc` and `.loc`

Two row selectors, mirroring the difference between *position* and *label*:

- `df.iloc[i]`  -> row by integer position (like list indexing; 0-based).
- `df.loc[label]` -> row by index label.

By default a DataFrame's index is `0, 1, 2, ...`, so `loc` and `iloc` look alike. They diverge once we set a meaningful index. Here we use the song title.

In [ ]:
# RUN-TOGETHER
# Position-based selection, while the index is still 0, 1, 2, ...
print('first row (iloc[0]):')
print(songs.iloc[0])
print()
print('first three rows (iloc[0:3]):')
print(songs.iloc[0:3][['title', 'streams_billions']])

In [ ]:
# RUN-TOGETHER
# Now replace the integer index with the song title.
songs_by_title = songs.set_index('title')
print(songs_by_title.head(3))

In [ ]:
# FILL-IN
# The SAME row, reached two ways: by position, and by label.
print('by position:', songs_by_title.iloc[0]['streams_billions'])
print('by label   :', songs_by_title.loc[____]['streams_billions'])   # use the LABEL

> **Discuss.** With the title as the index, `.loc['Riptide']` reads like a dictionary lookup, because that is essentially what it is. Which of the two selectors keeps working correctly if someone sorts the table?

### 3.3 Filtering rows with boolean masks

This is the same idea as NumPy boolean masks from Module 4, now on a whole table. A comparison on a column produces a column of `True`/`False`; putting that mask in the brackets keeps only the `True` rows.

Combine conditions with `&` (and) and `|` (or), and wrap each condition in parentheses.

In [ ]:
# RUN-TOGETHER
# Step 1: a comparison makes a boolean column.
mask = songs['streams_billions'] > 4
print(mask.head())

print()
# Step 2: use the mask to keep matching rows.
mega = songs[mask]
print('songs over 4 billion streams:', mega.shape[0])
mega[['title', 'streams_billions']]

In [ ]:
# RUN-TOGETHER
# Filter on text, and combine two conditions with & (parentheses required).
pop = songs[songs['genre'] == 'Pop']
print('Pop titles:', list(pop['title']))

recent_pop = songs[(songs['genre'] == 'Pop') & (songs['year'] >= 2020)]
recent_pop[['title', 'year']]

#### **Activity 6.4 - Now with `or`**

The previous cell used `&` (and). Now use `|` (or), and predict before you run.

Find songs that are *either* released before 2005 *or* have more than 4.5 billion streams. **Before running, guess how many rows you will get.**

In [ ]:
# FILL-IN  (Activity 6.4)
picks = songs[(songs['year'] < 2005) ____ (songs['streams_billions'] > 4.5)]
print('count:', picks.shape[0])
picks[['title', 'year', 'streams_billions']]

### 3.4 Sorting

`df.sort_values('column')` returns a new DataFrame ordered by that column. Add `ascending=False` for largest-first.

In [ ]:
# RUN-TOGETHER
# Most-streamed first.
top = songs.sort_values('streams_billions', ascending=False)
print(top[['title', 'streams_billions']].head())

print()
# Oldest songs first (ascending is the default).
print(songs.sort_values('year')[['title', 'year']].head())

#### **Activity 6.5 - Sorting on two keys**

**A.** Order by `genre` (A to Z), and *within* each genre by `streams_billions` (highest first). Pass a list of columns and a list of directions to `ascending`.

**B.** Using sort plus position, print the `title` of the *third* most-streamed song overall.

In [ ]:
# FILL-IN  (Activity 6.5)
# A. genre A-Z, then streams high-to-low within each genre
by_genre = songs.sort_values(['genre', 'streams_billions'], ascending=[____, ____])
print(by_genre[['genre', 'title', 'streams_billions']])

# B. third most-streamed overall: sort high-to-low, then take position 2 (0, 1, 2, ...)
third = songs.sort_values('streams_billions', ascending=False).iloc[____]
print('3rd most-streamed:', third['title'])

### 3.5 Creating new columns

We often need a value the raw data does not contain, computed from columns we do have. Assigning to a *new* column name creates it. Column math is element-wise (thanks to NumPy underneath), so no loop is needed.

In [ ]:
# RUN-TOGETHER
CURRENT_YEAR = 2026        # <-- update this line each year

# Age of each song, in years.
songs['age_years'] = CURRENT_YEAR - songs['year']

# A rate: average billions of streams PER year on the platform.
songs['streams_per_year'] = (songs['streams_billions'] / songs['age_years']).round(2)

# A boolean flag column.
songs['is_recent'] = songs['year'] >= 2020

songs[['title', 'year', 'age_years', 'streams_billions', 'streams_per_year', 'is_recent']].head()

#### **Activity 6.6 - Building a column to answer a question**

Create `gap_to_leader`: how far each song trails the top song (the **maximum** streams minus that song's streams). Then filter to the songs *within 1.0 billion* of the leader.

In [ ]:
# FILL-IN  (Activity 6.6)
leader = songs['streams_billions'].____()          # the max value
songs['gap_to_leader'] = (leader - songs['streams_billions']).round(1)

close = songs[songs['gap_to_leader'] ____ 1.0]     # within 1 billion of the top
close[['title', 'streams_billions', 'gap_to_leader']]

### 3.6 Summaries and descriptive statistics

A few workhorses:

- `df['col'].mean()`, `.median()`, `.min()`, `.max()`, `.sum()` -> one number about a column.
- `df['col'].value_counts()` -> counts of each category in a text column.
- `df.groupby('group_col')['value_col'].mean()` -> one summary *per group*.
- `df.describe()` -> count, mean, std, min, quartiles, max for every numeric column at once.

In [ ]:
# RUN-TOGETHER
print('mean streams (billions):  ', round(songs['streams_billions'].mean(), 2))
print('median streams (billions):', songs['streams_billions'].median())
print('newest release year:', songs['year'].max())
print()

print('songs per genre:')
print(songs['genre'].value_counts())

In [ ]:
# RUN-TOGETHER
# groupby: average streams per genre (one number per genre).
avg_by_genre = songs.groupby('genre')['streams_billions'].mean().round(2)
print(avg_by_genre.sort_values(ascending=False))

---
## 4. Capstone: which song is the "biggest"?
---

#### **Activity 6.7 - Two rankings, one dataset**

The station wants to name one song **"biggest of the era"** for a display in the student center. You have the table. This should be easy.

It is not, because "biggest" is not yet a question. At least two measures are defensible:

- **total streams** (`streams_billions`): how much listening the song has accumulated, ever.
- **streams per year** (`streams_per_year`): how fast it accumulates listening.

**Part A.** Print the top three songs by each measure.

In [ ]:
# FILL-IN  (Activity 6.7, Part A)
by_total = songs.sort_values(____, ascending=False)
print('TOP 3 BY TOTAL STREAMS')
print(by_total[['title', 'year', 'streams_billions']].head(3))

print()
by_rate = songs.sort_values(____, ascending=False)
print('TOP 3 BY STREAMS PER YEAR')
print(by_rate[['title', 'year', 'streams_per_year']].head(3))

**Part B.** The two lists do not agree, and they disagree even at number one. In a comment, explain *why* a 2024 release can top the per-year ranking while sitting nowhere near the top of the total ranking. Your answer should mention how long each song has been available.

In [ ]:
# Part B: your explanation as a comment
#

**Part C. The catch.** Look at `Bohemian Rhapsody` (1975). It is on this list.

Now think about what this table *is*. It is a list of the **most-streamed** songs. Answer in a comment:

1. Roughly how many songs were released in 1975 in total? Are they in this table?
2. So the 1975 songs in this table are what kind of 1975 song, compared to all 1975 songs?
3. If you compared "average streams of old songs" to "average streams of new songs" *using this table*, which group would be unfairly favored, and why?

In [ ]:
# Part C: your answers as comments
# 1.
# 2.
# 3.

**Part D. The recommendation.** Choose one measure, name the song, and defend the choice in one or two sentences. A good answer says what question your measure answers, and admits what it ignores.

In [ ]:
# Part D: your recommendation as a comment
#

> **Where this goes.** Part C is an instance of a **selection effect**: the table was assembled by choosing successful songs, so it cannot tell you how songs in general perform. This is the same failure you met in Module 4, where pooling the penguin species reversed a correlation, and it returns in Module 7 when we ask what a dataset is a sample *of*. A number computed over a hand-picked group describes the picking at least as much as the group.

## Suggested Exercises

1. A campus bike-share program tracks how many rides began at each station today:

    ```python
    rides = {'Union': 142, 'Library': 205, 'Rec Center': 88, 'Residence Hall': 176}
    ```

    a. Print the number of rides that began at the Library.

    b. A new station, `'Science Building'`, logged 61 rides. Add it.

    c. The Rec Center count was under-reported by 12. Correct it using its current value, without typing the new number.

    d. Write a loop that prints the total rides across all stations, and the name of the busiest station.

    e. Explain why a dictionary is a better fit here than two parallel lists.

2. Consider these three records:

    ```python
    trails = [
        {'name': 'Grandad Bluff', 'miles': 2.4, 'difficulty': 'moderate', 'dogs_ok': True},
        {'name': 'Hixon Forest',  'miles': 5.1, 'difficulty': 'hard',     'dogs_ok': True},
        {'name': 'Myrick Marsh',  'miles': 1.2, 'difficulty': 'easy',     'dogs_ok': False},
    ]
    ```

    a. Print the name of every trail under 3 miles.

    b. Build a DataFrame from this list of records. What are the column names, and where did they come from?

    c. Add a column `minutes` estimating hiking time at 25 minutes per mile, rounded to the nearest whole number.

    d. Which measurement scale (Module 2) is `difficulty`? What would go wrong if you called `.mean()` on it, and what summary should you use instead?

3. Using the `songs` DataFrame from this module:

    a. Write the expression that returns only the rows where the artist is `'Ed Sheeran'`.

    b. Write the expression for songs released between 2013 and 2019 inclusive. You will need two conditions and `&`.

    c. Use `groupby` to report the mean release year for each genre, sorted oldest first.

    d. `songs['genre'].value_counts()` shows nine genres for twelve songs. Explain why a `groupby` mean is fragile here, and what you would want before trusting the genre comparison.

4. A classmate writes this and is surprised by the result:

    ```python
    a = {'x': 1, 'y': 2}
    b = a
    b['z'] = 3
    print(a)
    ```

    a. What does `print(a)` show? Explain why.

    b. Which Module 3 idea is this, and what is the fix?

    c. Now suppose `a` were a DataFrame and `b = a` were followed by `b['new'] = ...`. Say why this class of bug is harder to notice in a 10,000-row table than in a two-key dictionary.